In [0]:
storage_account_name = "capstonestaccountadls"  # Replace with your storage account name
secret_scope = "keyvaultcapstone"
secret_key = "access-key"

# Retrieve the storage account key from the secret scope
storage_account_key = dbutils.secrets.get(scope=secret_scope, key=secret_key)

# Configure Spark to access ADLS Gen2 using the access key
spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
    storage_account_key
)

In [0]:
from pyspark.sql.functions import current_timestamp, lit
df_dates = spark.sql("""
    SELECT explode(sequence(to_date('1997-01-01'), current_date(), interval 1 day)) AS calendar_date
""")

In [0]:
df_bronze_calendar = (
    df_dates
    .withColumn("_ingestion_timestamp", current_timestamp())
    .withColumn("_source_system", lit("Spark_Generated"))
)

In [0]:
(df_bronze_calendar.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("mavenmarket.bronze.calendar")
)